In [1]:
from transformers import (
    TapasTokenizer, TapasForQuestionAnswering,
)
from datasets import load_dataset
import torch
import pandas as pd
from tqdm import tqdm

d:\Sorbonne\M2-MIND\MEDS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/wikitablequestions", trust_remote_code=True)
test_data = dataset["test"]

print("Number of test samples:", len(test_data))
print("Example:", test_data[0])


Number of test samples: 4344
Example: {'id': 'nu-0', 'question': 'which country had the most cyclists finish within the top 10?', 'answers': ['Italy'], 'table': {'header': ['Rank', 'Cyclist', 'Team', 'Time', 'UCI ProTour\\nPoints'], 'rows': [['1', 'Alejandro Valverde\xa0(ESP)', "Caisse d'Epargne", '5h 29\' 10"', '40'], ['2', 'Alexandr Kolobnev\xa0(RUS)', 'Team CSC Saxo Bank', 's.t.', '30'], ['3', 'Davide Rebellin\xa0(ITA)', 'Gerolsteiner', 's.t.', '25'], ['4', 'Paolo Bettini\xa0(ITA)', 'Quick Step', 's.t.', '20'], ['5', 'Franco Pellizotti\xa0(ITA)', 'Liquigas', 's.t.', '15'], ['6', 'Denis Menchov\xa0(RUS)', 'Rabobank', 's.t.', '11'], ['7', 'Samuel Sánchez\xa0(ESP)', 'Euskaltel-Euskadi', 's.t.', '7'], ['8', 'Stéphane Goubert\xa0(FRA)', 'Ag2r-La Mondiale', '+ 2"', '5'], ['9', 'Haimar Zubeldia\xa0(ESP)', 'Euskaltel-Euskadi', '+ 2"', '3'], ['10', 'David Moncoutié\xa0(FRA)', 'Cofidis', '+ 2"', '1']], 'name': 'csv/203-csv/733.tsv'}}


In [16]:
tapas_tokenizer = TapasTokenizer.from_pretrained("google/tapas-large-finetuned-wtq")
tapas_model = TapasForQuestionAnswering.from_pretrained("google/tapas-large-finetuned-wtq")


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
d:\Sorbonne\M2-MIND\MEDS\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\titou\.cache\huggingface\hub\models--google--tapas-large-finetuned-wtq. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this 

The code below is a basic test with like 30% accuracy but the paper says it can reach 48%. And we used the wtq finetuned for this test so the problem is most likely coming from us (except if the base model doesnt perform as well i need to double check that). Anyways maybe a better config could do the trick, see here: https://huggingface.co/docs/transformers/model_doc/tapas

In [ ]:
def normalize_answer(a):# normalize answers to try and avoid errors because of mismatching
    if a is None:
        return "" # default
    a = str(a).strip().lower() 
    # remove commas
    a = a.replace(",", "")
    # handle percentages
    if a.endswith("%"):
        try:
            return float(a[:-1]) / 100 # turn into a normal float
        except:
            return a
    # convert numeric strings to float
    try:
        return float(a)
    except:
        return a

In [20]:
def compute_final_answer(table, answer_coords, aggregation):
    """
    Convert selected cell coordinates + aggregation into final predicted answer.
    """
    if not answer_coords:
        return None
    
    values = [table.iat[row, col] for row, col in answer_coords]
    
    if aggregation == "NONE":
        return " ".join(map(str, values))
    elif aggregation == "COUNT":
        return len(values)
    elif aggregation == "SUM":
        try:
            return sum(float(v) for v in values)
        except:
            return " ".join(map(str, values))
    elif aggregation == "AVERAGE":
        try:
            return sum(float(v) for v in values) / len(values)
        except:
            return " ".join(map(str, values))
    else:
        return " ".join(map(str, values))

In [21]:
def test_model(model, tokenizer, data):
    correct = 0
    total = len(data)
    total_skipped = 0
    print("Evaluating on", total, "samples...")

    progress = tqdm(data, desc="Evaluating", ncols=100)

    for ex in progress:
        headers = ex["table"]["header"]
        rows = ex["table"]["rows"]

        # handle irregular row lengths
        max_len = len(headers)
        clean_rows = [r + [""] * (max_len - len(r)) for r in rows]
        table = pd.DataFrame(clean_rows, columns=headers)
        question = ex["question"]

        # Tokenize 
        inputs = tokenizer(
            table=table,
            queries=[question],
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        # Run model
        try:
            with torch.no_grad():
                outputs = model(**inputs)
        except IndexError:
            total_skipped += 1
            progress.set_postfix({
                "accuracy": f"{(correct / max(1, (progress.n - total_skipped))) * 100:.2f}%",
                "skipped": total_skipped
            })
            continue
        logits = outputs.logits.cpu()

        predictions = tokenizer.convert_logits_to_predictions(inputs, logits)
        if len(predictions) == 2:
            predicted_answer_coordinates, predicted_aggregation = predictions
        else:
            predicted_answer_coordinates = predictions[0]
            predicted_aggregation = ["NONE"]

        # Compute final answer using aggregation
        predicted_answer = compute_final_answer(table, predicted_answer_coordinates[0], predicted_aggregation[0])

        # Normalize predicted and gold answers
        normalized_pred = normalize_answer(predicted_answer)
        normalized_gold = set(map(normalize_answer, ex["answers"]))

        if normalized_pred in normalized_gold:
            correct += 1

        progress.set_postfix({
            "accuracy": f"{(correct / max(1, (progress.n - total_skipped))) * 100:.2f}%",
            "skipped": total_skipped
        })

    return correct, total, total_skipped

In [22]:
dev_test = dataset["validation"] # this is what they use in the paper for the accuracy

print("Number of test samples:", len(dev_test))

Number of test samples: 2831


In [23]:
correct, total, total_skipped = test_model(tapas_model, tapas_tokenizer, dev_test)
print(f"Accuracy: {correct / (total - total_skipped) * 100:.2f}%, out of {total - total_skipped} samples (skipped {total_skipped})")

Evaluating on 2831 samples...


Evaluating: 100%|███████████████████| 2831/2831 [32:21<00:00,  1.46it/s, accuracy=32.44%, skipped=6]

Accuracy: 32.42%, out of 2825 samples (skipped 6)
